In [1]:
pip install rapidfuzz

  Using cached rapidfuzz-3.12.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached rapidfuzz-3.12.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [66]:
import pandas as pd
import geopandas as gpd
import folium
from geopy.distance import geodesic
from rapidfuzz import process, fuzz
import matplotlib.pyplot as plt

In [3]:
def display_schools(file_path, output_path):
    """
    Reads a CSV file, filters out any rows where 'campus name' contains 'Lowell' or 'Ruth Awasa', 
    and writes to a new CSV file.
    :param file_path: Path to the CSV file.
    :param output_path: Path to save the filtered CSV file.
    """
    try:
        # Load the CSV file
        df = pd.read_csv('Schools_20250202.csv')
        
        # Ensure the column name is correct (update if necessary)
        if 'Campus Name' not in df.columns:
            raise ValueError("Column 'campus name' not found in CSV file. Please check the column names.")
        
        # Filter out schools where 'campus name' contains the excluded words
        excluded_keywords = ["Lowell", "Asawa"]
        filtered_df = df[~df['Campus Name'].str.contains('|'.join(excluded_keywords), case=False, na=False)]
        
        # Save to a new CSV file
        filtered_df.to_csv(output_path, index=False)
        
        # Display the remaining schools
        print(filtered_df)
        print(f"Filtered data has been saved to {output_path}")
    except Exception as e:
        print("Error:", e)

In [4]:
Filtered = gpd.read_file('Filtered_Schools.csv')

In [5]:
Filtered.head()

,Campus Name,CCSF Entity,Lower Grade,Upper Grade,Grade Range,Category,Map Label,Lower Age,Upper Age,General Type,...,Campus Address,Supervisor District,County FIPS,County Name,Location 1,Neighborhoods (old),Zip Codes,Fire Prevention Districts,Police Districts,Supervisor Districts
0,"Milk, Harvey Milk Childrens Center",SFUSD,-2,-1,PK,USD PreK,CDC095,3,4,CDC,...,"841 ELLIS ST, SAN FRANCISCO CA 94117",6,6075,SAN FRANCISCO,"CA\n(37.783802, -122.420105)",36,28858,7.0,9.0,9
1,Mckinley Elementary School,SFUSD,0,5,K-5,USD Grades K-5,PS075,5,10,PS,...,"1025 14TH ST, San Francisco, CA 94114",8,6075,SAN FRANCISCO,"CA\n(37.766884, -122.436279)",3,28862,15.0,5.0,5
2,Jewish Community Center San Francisco - Rosenb...,Private,-2,-1,PK,Independent / Private,CDC058,3,4,CDC,...,"325 ARGUELLO BLVD, SAN FRANCISCO, CA 94118",1,6075,SAN FRANCISCO,"CA\n(37.784588, -122.459488)",11,54,11.0,6.0,2
3,Eureka Learning Center,Private,-2,-1,PK,Independent / Private,CDC035,3,4,CDC,...,"464 DIAMOND ST, SAN FRANCISCO, CA 94114",8,6075,SAN FRANCISCO,"CA\n(37.754967, -122.437004)",22,28862,2.0,4.0,5
4,Noriega Early Education School,SFUSD,-2,5,PK-5,USD PreK/TK-5,PS085,3,10,PS,...,"1775 44TH AVE, San Francisco, CA 94122",4,6075,SAN FRANCISCO,"CA\n(37.753834, -122.503654)",35,56,1.0,8.0,3


In [6]:
publicschools = Filtered[Filtered['CCSF Entity'].str.contains('SFUSD')]

In [7]:
publicschools

,Campus Name,CCSF Entity,Lower Grade,Upper Grade,Grade Range,Category,Map Label,Lower Age,Upper Age,General Type,...,Campus Address,Supervisor District,County FIPS,County Name,Location 1,Neighborhoods (old),Zip Codes,Fire Prevention Districts,Police Districts,Supervisor Districts
0,"Milk, Harvey Milk Childrens Center",SFUSD,-2,-1,PK,USD PreK,CDC095,3,4,CDC,...,"841 ELLIS ST, SAN FRANCISCO CA 94117",6,6075,SAN FRANCISCO,"CA\n(37.783802, -122.420105)",36,28858,7.0,9.0,9
1,Mckinley Elementary School,SFUSD,0,5,K-5,USD Grades K-5,PS075,5,10,PS,...,"1025 14TH ST, San Francisco, CA 94114",8,6075,SAN FRANCISCO,"CA\n(37.766884, -122.436279)",3,28862,15.0,5.0,5
4,Noriega Early Education School,SFUSD,-2,5,PK-5,USD PreK/TK-5,PS085,3,10,PS,...,"1775 44TH AVE, San Francisco, CA 94122",4,6075,SAN FRANCISCO,"CA\n(37.753834, -122.503654)",35,56,1.0,8.0,3
10,Sutro Elementary / Early Education,SFUSD,0,5,K-5,USD Grades K-5,PS113,5,10,PS,...,"235 12TH AVE, San Francisco, CA 94118",1,6075,SAN FRANCISCO,"CA\n(37.783604, -122.471451)",11,54,11.0,6.0,2
11,"Marshall, Thurgood Marshall High School",SFUSD,9,12,9-12,USD Grades 9-12,PS073,14,17,PS,...,"45 CONKLING ST, San Francisco, CA 94124",10,6075,SAN FRANCISCO,"CA\n(37.736309, -122.401649)",1,58,10.0,3.0,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434,Longfellow Elementary School,SFUSD,0,5,K-5,USD Grades K-5,PS068,5,10,PS,...,"755 MORSE ST, San Francisco, CA 94112",11,6075,SAN FRANCISCO,"CA\n(37.710423, -122.446739)",5,28861,9.0,7.0,6
435,"Webster, Daniel Webster Out-Of-School (OST) Pr...",SFUSD,0,5,K-5,USD Grades K-5,PS123,5,10,PS,...,"465 MISSOURI ST, San Francisco, CA 94107",10,6075,SAN FRANCISCO,"CA\n(37.760521, -122.395843)",29,28856,10.0,3.0,8
436,Mission High School,SFUSD,9,12,9-12,USD Grades 9-12,PS080,14,17,PS,...,"3750 18TH ST, San Francisco, CA 94114",8,6075,SAN FRANCISCO,"CA\n(37.76202, -122.427307)",3,28862,8.0,4.0,5
437,Miraloma Elementary School,SFUSD,0,5,K-5,USD Grades K-5,PS078,5,10,PS,...,"175 OMAR WAY, San Francisco, CA 94127",7,6075,SAN FRANCISCO,"CA\n(37.738636, -122.450188)",40,59,9.0,7.0,4


In [8]:
publichighschools = publicschools[publicschools['Grade Range'].str.contains('9-12')]

In [9]:
publichighschools

,Campus Name,CCSF Entity,Lower Grade,Upper Grade,Grade Range,Category,Map Label,Lower Age,Upper Age,General Type,...,Campus Address,Supervisor District,County FIPS,County Name,Location 1,Neighborhoods (old),Zip Codes,Fire Prevention Districts,Police Districts,Supervisor Districts
11,"Marshall, Thurgood Marshall High School",SFUSD,9,12,9-12,USD Grades 9-12,PS073,14,17,PS,...,"45 CONKLING ST, San Francisco, CA 94124",10,6075,SAN FRANCISCO,"CA\n(37.736309, -122.401649)",1,58,10.0,3.0,8
38,"Hearst, Phoebe Apperson Hearst Home",SFUSD,9,12,9-12,USD Grades 9-12,PS045,14,17,PS,...,"3045 SANTIAGO ST, SAN FRANCISCO 94116",4,6075,SAN FRANCISCO,"CA\n(37.74363, -122.500053)",35,29491,1.0,8.0,3
90,"Burton, Phillip And Sala Burton High School",SFUSD,9,12,9-12,USD Grades 9-12,PS011,14,17,PS,...,"400 MANSELL ST, San Francisco, CA 94134",9,6075,SAN FRANCISCO,"CA\n(37.721546, -122.406555)",28,309,10.0,3.0,7
92,"Washington, George Washington High School",SFUSD,9,12,9-12,USD Grades 9-12,PS121,14,17,PS,...,"600 32ND AVE, San Francisco, CA 94121",1,6075,SAN FRANCISCO,"CA\n(37.777905, -122.491013)",26,55,11.0,6.0,2
135,"Lincoln, Abraham Lincoln High School",SFUSD,9,12,9-12,USD Grades 9-12,PS067,14,17,PS,...,"2162 24TH AVE, San Francisco, CA 94116",4,6075,SAN FRANCISCO,"CA\n(37.746594, -122.48024)",35,29491,1.0,8.0,3
174,Life Learning Academy Charter School,SFUSD,9,12,9-12,USD Charter School,PS064,14,17,PS,...,"651 8TH TI ST, SAN FRANCISCO, CA 94130",6,6075,SAN FRANCISCO,"CA\n(37.825512, -122.367996)",37,62,,2.0,9
194,Gateway High School / Kipp Sf Bay Academy,SFUSD,9,12,9-12,USD Charter School,PS037,14,17,PS,...,"1430 SCOTT ST, San Francisco, CA 94115",5,6075,SAN FRANCISCO,"CA\n(37.783264, -122.436691)",41,29490,13.0,5.0,11
195,Galileo High School,SFUSD,9,12,9-12,USD Grades 9-12,PS035,14,17,PS,...,"1150 FRANCISCO ST, San Francisco, CA 94109",2,6075,SAN FRANCISCO,"CA\n(37.803791, -122.424149)",32,28858,5.0,9.0,1
219,Balboa High School,SFUSD,9,12,9-12,USD Grades 9-12,PS007,14,17,PS,...,"1000 CAYUGA AVE, San Francisco, CA 94112",11,6075,SAN FRANCISCO,"CA\n(37.721142, -122.441399)",25,28861,9.0,7.0,6
284,City Arts And Tech High School,SFUSD,9,12,9-12,USD Grades 9-12,PS019,14,17,PS,...,"325 LA GRANDE AVE, San Francisco, CA 94112",11,6075,SAN FRANCISCO,"CA\n(37.718784, -122.424667)",18,309,9.0,7.0,6


In [10]:
publichighschools['Campus Name'].unique()

array(['Marshall, Thurgood Marshall High School',
       'Hearst, Phoebe Apperson Hearst Home',
       'Burton, Phillip And Sala Burton High School',
       'Washington, George Washington High School',
       'Lincoln, Abraham Lincoln High School',
       'Life Learning Academy Charter School',
       'Gateway High School / Kipp Sf Bay Academy', 'Galileo High School',
       'Balboa High School', 'City Arts And Tech High School',
       'Independence High School',
       'Wallenberg, Raoul Wallenberg High School', 'Downtown High School',
       'San Francisco International High School',
       'Jordan, June Jordan High School',
       "O'Connell, John O'Connell High School", 'Mission High School',
       'Wells, Ida B. Wells High School'], dtype=object)

In [11]:
enrollment = pd.read_csv("cdenroll2324-v24.csv") 

In [12]:
dataframe2 = enrollment.dropna()

In [13]:
dataframe2

,AcademicYear,AggregateLevel,CountyCode,DistrictCode,SchoolCode,CountyName,DistrictName,SchoolName,Charter,ReportingCategory,...,GR_03,GR_04,GR_05,GR_06,GR_07,GR_08,GR_09,GR_10,GR_11,GR_12
272,2023-24,S,1,10017.0,112607.0,Alameda,Alameda County Office of Education,Envision Academy for Arts & Technology,Y,AR_0418,...,0,0,0,23,11,15,34,36,53,50
273,2023-24,S,1,10017.0,112607.0,Alameda,Alameda County Office of Education,Envision Academy for Arts & Technology,Y,AR_1922,...,0,0,0,0,0,0,0,0,0,1
274,2023-24,S,1,10017.0,112607.0,Alameda,Alameda County Office of Education,Envision Academy for Arts & Technology,Y,ELAS_EL,...,*,*,*,*,*,*,*,*,*,15
275,2023-24,S,1,10017.0,112607.0,Alameda,Alameda County Office of Education,Envision Academy for Arts & Technology,Y,ELAS_EO,...,*,*,*,20,*,*,21,16,26,19
276,2023-24,S,1,10017.0,112607.0,Alameda,Alameda County Office of Education,Envision Academy for Arts & Technology,Y,ELAS_IFEP,...,*,*,*,*,*,*,*,*,*,*
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267757,2023-24,S,58,72769.0,5838305.0,Yuba,Wheatland Union High,Wheatland Union High,N,SG_FS,...,*,*,*,*,*,*,*,*,*,*
267758,2023-24,S,58,72769.0,5838305.0,Yuba,Wheatland Union High,Wheatland Union High,N,SG_HM,...,*,*,*,*,*,*,*,*,*,*
267759,2023-24,S,58,72769.0,5838305.0,Yuba,Wheatland Union High,Wheatland Union High,N,SG_MG,...,*,*,*,*,*,*,*,*,*,*
267760,2023-24,S,58,72769.0,5838305.0,Yuba,Wheatland Union High,Wheatland Union High,N,SG_SD,...,*,*,*,*,*,*,199,200,180,221


In [14]:
dataframe3 = dataframe2[dataframe2['CountyName'].str.contains('San Francisco', )]

In [15]:
df_excluded = dataframe3[~dataframe3['GR_09'].str.contains('\*')]

In [16]:
dataframe5 = df_excluded[df_excluded['DistrictName'].str.contains('San Francisco Unified')]

In [17]:
dataframe6 = dataframe5[~dataframe5['SchoolName'].str.contains('Elementary', 'Middle')]

In [18]:
races = r"RE_A|RE_B|RE_D|RE_F|RE_H|RE_I|RE_P|RE_T|RE_W"

In [19]:
dataframe8 = dataframe6[dataframe6['ReportingCategory'].str.contains(races, na=False)]

In [20]:
unique_values = dataframe8['SchoolName'].unique()
unique_values

array(['District Office', 'KIPP Bayview Academy',
       'KIPP San Francisco Bay Academy',
       "Five Keys Charter (SF Sheriff's)",
       'Jordan (June) School for Equity',
       'City Arts & Leadership Academy',
       'Stockton (Commodore) Children Center', 'Noriega Children Center',
       'Tule Elk Park Children Center', 'McLaren (John) Children Centers',
       "Five Keys Independence HS (SF Sheriff's)",
       'S.F. International High', 'Academy (The)- SF @McAteer',
       'Chinese Immersion School at DeAvila',
       'San Francisco Public Montessori', 'Gateway Middle',
       'Mission Preparatory', 'KIPP San Francisco College Preparatory',
       'Brown Jr. (Willie L) Middle', 'Wells (Ida B.) High',
       'Downtown High', 'Independence High',
       'Wallenberg (Raoul) Traditional High',
       'Burton (Phillip and Sala) Academic High', 'Balboa High',
       'Asawa (Ruth) SF Sch of the Arts, A Public School',
       'Marshall (Thurgood) High', 'Life Learning Academy Charter

In [21]:
# Function to find similar names with debug
def find_similar_names(name, name_list, threshold=51):
    matches = process.extract(name, name_list, scorer=fuzz.ratio, limit=1)
    if matches:
        print(f"Checking '{name}' against '{matches[0][0]}' with score {matches[0][1]}")
        if matches[0][1] >= threshold:
            return True
    return False

In [22]:
# Get the list of public high school names
public_high_school_names = publichighschools['Campus Name'].tolist()

In [23]:
public_high_school_names2 = dataframe8['SchoolName'].tolist()

In [24]:
similar_names = dataframe8['SchoolName'].apply(lambda x: find_similar_names(x, public_high_school_names))

Checking 'District Office' against 'Mission High School' with score 35.29411764705882
Checking 'District Office' against 'Mission High School' with score 35.29411764705882
Checking 'District Office' against 'Mission High School' with score 35.29411764705882
Checking 'District Office' against 'Mission High School' with score 35.29411764705882
Checking 'District Office' against 'Mission High School' with score 35.29411764705882
Checking 'District Office' against 'Mission High School' with score 35.29411764705882
Checking 'District Office' against 'Mission High School' with score 35.29411764705882
Checking 'KIPP Bayview Academy' against 'Gateway High School / Kipp Sf Bay Academy' with score 42.622950819672134
Checking 'KIPP Bayview Academy' against 'Gateway High School / Kipp Sf Bay Academy' with score 42.622950819672134
Checking 'KIPP Bayview Academy' against 'Gateway High School / Kipp Sf Bay Academy' with score 42.622950819672134
Checking 'KIPP Bayview Academy' against 'Gateway High Sc

In [70]:
filtered_df_based_on_similar_names = dataframe8[similar_names]
filtered_df_based_on_similar_names.head()

,AcademicYear,AggregateLevel,CountyCode,DistrictCode,SchoolCode,CountyName,DistrictName,SchoolName,Charter,ReportingCategory,...,GR_03,GR_04,GR_05,GR_06,GR_07,GR_08,GR_09,GR_10,GR_11,GR_12
196060,2023-24,S,38,68478.0,102103.0,San Francisco,San Francisco Unified,Jordan (June) School for Equity,N,RE_A,...,0,0,0,0,0,0,5,4,0,0
196061,2023-24,S,38,68478.0,102103.0,San Francisco,San Francisco Unified,Jordan (June) School for Equity,N,RE_B,...,0,0,0,0,0,0,6,4,4,4
196062,2023-24,S,38,68478.0,102103.0,San Francisco,San Francisco Unified,Jordan (June) School for Equity,N,RE_D,...,0,0,0,0,0,0,3,2,1,0
196063,2023-24,S,38,68478.0,102103.0,San Francisco,San Francisco Unified,Jordan (June) School for Equity,N,RE_F,...,0,0,0,0,0,0,3,1,2,4
196064,2023-24,S,38,68478.0,102103.0,San Francisco,San Francisco Unified,Jordan (June) School for Equity,N,RE_H,...,0,0,0,0,0,0,34,31,43,39


In [88]:
# Filter the DataFrame to include only the specified grades in ReportingCategory
grades = ['GR_09', 'GR_10', 'GR_11', 'GR_12']
filtered_df_based_on_similar_names2 = filtered_df_based_on_similar_names[['SchoolName', 'ReportingCategory'] + grades]
filtered_df_based_on_similar_names2

,SchoolName,ReportingCategory,GR_09,GR_10,GR_11,GR_12


In [82]:
grouped_df = filtered_df_based_on_similar_names.groupby(['SchoolName', 'ReportingCategory']).size().unstack(fill_value=0)


In [83]:
# Create a bar chart
plt.figure(figsize=(15, 10))
grouped_df.plot(kind='bar', stacked=True, colormap='tab20', figsize=(15, 10))
plt.title('Distribution by School Names and Reporting Category')
plt.xlabel('School Name')
plt.ylabel('Total Count')  # Adjust y-axis label to indicate total numbers
plt.xticks(rotation=45, ha='right')
plt.legend(title='Reporting Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

TypeError: no numeric data to plot

<Figure size 1500x1000 with 0 Axes>

In [26]:
stops = gpd.read_file('Muni_Stops_20250126.csv')

In [27]:
routes = gpd.read_file('Muni_Simple_Routes_20250202.csv')

In [28]:
route2df = pd.read_csv('routes.txt')

In [29]:
trips2 = pd.read_csv('trips.txt')

In [30]:
stoptimes = pd.read_csv('stop_times.txt')

In [45]:
filtered_stops_df = pd.DataFrame(filtered_stops)

In [46]:
stopslist = filtered_stops_df['STOPID'].tolist()

In [47]:
sf_map = folium.Map(location=[37.7749, -122.4194], zoom_start=12)

In [48]:
stoptimes['stop_id'] = stoptimes['stop_id'].astype(str)
stopslist = [str(stop) for stop in stopslist]

stoptimesatbus = stoptimes[stoptimes['stop_id'].isin(stopslist)]

In [49]:
tripids = stoptimesatbus['trip_id'].tolist()

In [50]:
trips3 = trips2[trips2['trip_id'].isin(tripids)]

In [51]:
trips4 = trips3['route_id'].tolist()

In [58]:
onlyroutes = routes[routes['ROUTE_NAME'].isin(trips4)]

In [59]:
# Function to calculate distance between two coordinates
def calculate_distance(coord1, coord2):
    return geodesic(coord1, coord2).miles

In [60]:
# Function to extract and clean coordinates from the 'Location 1' field
def extract_coordinates(location):
    try:
        # Assuming the location is a string of format "CA\n(latitude, longitude)"
        parts = location.split('(')[1].strip(')').split(',')
        # Convert to float and return as a tuple
        return float(parts[0]), float(parts[1])
    except (ValueError, IndexError):
        # If conversion fails, return None
        return None

In [61]:
# Filter stops within 0.25 miles of any school
filtered_stops = []

for stop_idx, stop_row in stops.iterrows():
    stop_coord = (stop_row['LATITUDE'], stop_row['LONGITUDE'])
    for school_idx, school_row in publichighschools.iterrows():
        # Extract school coordinates
        school_coord = extract_coordinates(school_row['Location 1'])
        if school_coord:
            # Calculate distance
            distance = calculate_distance(stop_coord, school_coord)
            if distance <= 0.25:
                filtered_stops.append(stop_row)
                break

In [62]:
# Add filtered Muni stops to the map
for idx, row in filtered_stops_df.iterrows():
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=3,  # Radius of the circle marker
        color='blue',  # Border color of the circle marker
        fill=True,  # Fill the circle marker
        fill_color='blue'  # Fill color of the circle marker
    ).add_to(sf_map)

In [63]:
# Add schools to the map with red color
for idx, row in publichighschools.iterrows():
    school_coord = extract_coordinates(row['Location 1'])
    if school_coord:
        folium.CircleMarker(
            location=school_coord,
            radius=3,  # Radius of the circle marker
            color='green',  # Border color of the circle marker
            fill=True,  # Fill the circle marker
            fill_color='green',  # Fill color of the circle marker
            popup=row['Campus Name']  # Popup with school name
        ).add_to(sf_map)

In [64]:
for idx, row in onlyroutes.iterrows():
    # Extract the route shape from the 'shape' column
    shape = row['shape'].replace('MULTILINESTRING ((', '').replace('))', '')
    points = shape.split(', ')
    coordinates = [(float(point.split()[1]), float(point.split()[0])) for point in points]
    
    folium.PolyLine(
        coordinates,
        color='red',
        weight=2.5,
        opacity=1
    ).add_to(sf_map)

In [65]:
sf_map